# CellLineFinder — Unified Data Loader (Batch 1)

This notebook wraps `data_loader.py`, which harmonises the **Batch 1** data
sources onto a single key — `DepMap_ID` (`ACH-xxxxxx`) — so a single gene
query can pull RNA expression, protein expression, mutations and fusions
together.

**Batch 1 sources covered:**

| File | Content | Role |
|---|---|---|
| 9 | `DepMap_sample_info.csv` | Master cell line table (the ID hub) |
| 8 | `DepMap_OmicsProfiles.csv` | ProfileID/SequencingID → ACH-ID bridge |
| 2 | `DepMap_OmicsExpression...csv` | mRNA expression (log2 TPM+1) |
| 4 | `Harmonized_MS_CCLE_Gygi...csv` | Protein expression (log2 MS intensity) |
| 6 | `OmicsSomaticMutationsProfile.csv` | Somatic mutations (exclusion criteria) |
| 5 | `OmicsFusionFilteredSupplementary.csv` | Gene fusions (exclusion criteria) |

**Folder assumption:** this notebook lives in
`Data Science MSc Graduation Project/`, with the data in
`Data Science MSc Graduation Project/data-2/` (matching your current Finder
layout: `data-2/gene expression/`, `data-2/gene properties/`, `data-2/nomenclature/`).

> `data_loader.py` must sit next to this notebook (or be importable on `sys.path`).

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Make sure data_loader.py (in the same folder as this notebook) is importable
sys.path.append(str(Path.cwd()))

from data_loader import CellLineDataLoader
import pandas as pd

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 140)

In [2]:
# Point this at your data-2 folder.
# If this notebook and data-2 are siblings (as in your current setup), "data-2" is enough.
DATA_DIR = "data-2"

loader = CellLineDataLoader(DATA_DIR)
print("Loader initialised. Data directory:", loader.data_dir.resolve())

Loader initialised. Data directory: /Users/liyannan/Desktop/Data Science MSc Graduation Project/data-2


## 2. Sanity check: master sample table (File 9)

This is the hub every other table gets joined back to.

In [3]:
sample_info = loader.sample_info
print(f"Shape: {sample_info.shape}")
sample_info[["DepMap_ID", "cell_line_name", "lineage", "primary_disease"]].head()

Shape: (1840, 29)


,DepMap_ID,cell_line_name,lineage,primary_disease
0,ACH-000016,SLR 21,kidney,Kidney Cancer
1,ACH-000032,MHH-CALL-3,blood,Leukemia
2,ACH-000033,NCI-H1819,lung,Lung Cancer
3,ACH-000043,Hs 895.T,fibroblast,Non-Cancerous
4,ACH-000049,HEK TE,kidney,Non-Cancerous


## 3. Single-layer queries

Each method below returns a tidy DataFrame indexed by `DepMap_ID`, so you
can inspect one omics layer at a time before combining them.

In [4]:
GENE = "EGFR"   # change this to explore any gene

rna = loader.get_rna_expression(GENE)
print(f"RNA expression rows: {len(rna)}")
rna.sort_values("rna_expression", ascending=False).head()

RNA expression rows: 1495


,DepMap_ID,rna_expression
493,ACH-001523,10.531869
802,ACH-000741,10.076148
251,ACH-000012,9.673645
632,ACH-001411,9.417620
587,ACH-000637,9.286535


In [5]:
protein = loader.get_protein_expression(GENE)
print(f"Protein expression rows: {len(protein)} (NaN = not detected by MS)")
protein.sort_values("protein_expression", ascending=False).head()

Protein expression rows: 375 (NaN = not detected by MS)


,DepMap_ID,protein_expression
329,ACH-000741,5.154394
88,ACH-000916,4.987237
214,ACH-000433,3.617746
82,ACH-000454,3.454519
273,ACH-000693,3.443853


In [6]:
mutations = loader.get_mutations(GENE)
print(f"Mutation records: {len(mutations)}")
mutations.head()

[data_loader] Warning: 152885 of 1066869 mutation records could not be mapped to an ACH-ID (ProfileID missing from file 8). These rows are dropped.
Mutation records: 151


,DepMap_ID,ProteinChange,VariantType,VepImpact,AF,DP,is_lof
0,ACH-000784,p.Y69MfsTer11,deletion,HIGH,0.580,36,True
1,ACH-000515,p.Q71L,SNV,MODERATE,0.194,39,False
2,ACH-002215,p.R108K,SNV,MODERATE,0.342,84,False
3,ACH-001526,p.D191Y,SNV,MODERATE,0.467,44,False
4,ACH-000955,p.V292M,SNV,MODERATE,0.472,36,False


In [7]:
fusions = loader.get_fusions(GENE)
print(f"Fusion event records: {len(fusions)}")
fusions.head()

Fusion event records: 48


,DepMap_ID,CanonicalFusionName,FFPM,confidence,reading_frame
0,ACH-001864,CDC42SE1--EGFR,0.296298,high,out-of-frame
1,ACH-001864,CDC42SE1--EGFR,0.253970,high,out-of-frame
2,ACH-001864,CDC42SE1--EGFR,0.253970,medium,out-of-frame
3,ACH-001864,CDC42SE1--EGFR,0.211641,medium,out-of-frame
4,ACH-000785,EGFR--UPP1,0.024209,low,in-frame


## 4. Combined query — the actual CellLineFinder output

`build_gene_profile()` is the function that matters: one call merges RNA,
protein, mutation and fusion evidence for a target gene across every cell
line, and (optionally) flags cell lines to exclude based on a second gene
list (e.g. known resistance mutations).

In [8]:
profile = loader.build_gene_profile(GENE)

print(f"Cell lines with at least one piece of evidence for {GENE}: {len(profile)}")
profile.head(15)

Cell lines with at least one piece of evidence for EGFR: 1434


,DepMap_ID,cell_line_name,lineage,primary_disease,rna_expression,protein_expression,has_target_mutation,has_target_fusion,excluded
0,ACH-001523,HSC-1,skin,Skin Cancer,10.531869,NaN,False,False,False
1,ACH-000741,U-BLC1,urinary_tract,Bladder Cancer,10.076148,5.154394,False,True,False
2,ACH-000012,HCC827,lung,Lung Cancer,9.673645,2.724113,True,False,False
3,ACH-001411,UM-UC-5,urinary_tract,Bladder Cancer,9.417620,NaN,False,False,False
4,ACH-000637,KYSE-520,esophagus,Esophageal Cancer,9.286535,NaN,False,False,False
5,ACH-001864,YSCCC,bile_duct,Bile Duct Cancer,9.258024,NaN,False,True,False
6,ACH-000849,MDA-MB-468,breast,Breast Cancer,9.182568,2.649190,False,True,False
7,ACH-000109,NCI-H3255,lung,Lung Cancer,8.925347,2.465903,True,False,False
8,ACH-002680,170-MG-BA,central_nervous_system,Brain Cancer,8.867155,NaN,False,True,False
9,ACH-001649,Shmac 5,prostate,Prostate Cancer,8.362821,NaN,False,False,False


### With exclusion criteria

Example: find cell lines with high EGFR expression, but flag/exclude ones
that also carry a KRAS mutation or fusion (a common resistance-pathway
exclusion in EGFR-targeted therapy research).

In [9]:
profile_excl = loader.build_gene_profile(GENE, exclude_genes=["KRAS"])

print(f"Total: {len(profile_excl)}  |  Flagged for exclusion: {profile_excl['excluded'].sum()}")
profile_excl.head(15)

Total: 1434  |  Flagged for exclusion: 237


,DepMap_ID,cell_line_name,lineage,primary_disease,rna_expression,protein_expression,has_target_mutation,has_target_fusion,excluded
0,ACH-001523,HSC-1,skin,Skin Cancer,10.531869,NaN,False,False,False
1,ACH-000741,U-BLC1,urinary_tract,Bladder Cancer,10.076148,5.154394,False,True,False
2,ACH-000012,HCC827,lung,Lung Cancer,9.673645,2.724113,True,False,False
3,ACH-001411,UM-UC-5,urinary_tract,Bladder Cancer,9.417620,NaN,False,False,False
4,ACH-000637,KYSE-520,esophagus,Esophageal Cancer,9.286535,NaN,False,False,False
5,ACH-001864,YSCCC,bile_duct,Bile Duct Cancer,9.258024,NaN,False,True,False
6,ACH-000849,MDA-MB-468,breast,Breast Cancer,9.182568,2.649190,False,True,False
7,ACH-000109,NCI-H3255,lung,Lung Cancer,8.925347,2.465903,True,False,False
8,ACH-002680,170-MG-BA,central_nervous_system,Brain Cancer,8.867155,NaN,False,True,True
9,ACH-001649,Shmac 5,prostate,Prostate Cancer,8.362821,NaN,False,False,False


In [10]:
# Top candidates after removing excluded cell lines
clean_candidates = profile_excl.loc[~profile_excl["excluded"]]
clean_candidates.head(10)

,DepMap_ID,cell_line_name,lineage,primary_disease,rna_expression,protein_expression,has_target_mutation,has_target_fusion,excluded
0,ACH-001523,HSC-1,skin,Skin Cancer,10.531869,NaN,False,False,False
1,ACH-000741,U-BLC1,urinary_tract,Bladder Cancer,10.076148,5.154394,False,True,False
2,ACH-000012,HCC827,lung,Lung Cancer,9.673645,2.724113,True,False,False
3,ACH-001411,UM-UC-5,urinary_tract,Bladder Cancer,9.417620,NaN,False,False,False
4,ACH-000637,KYSE-520,esophagus,Esophageal Cancer,9.286535,NaN,False,False,False
5,ACH-001864,YSCCC,bile_duct,Bile Duct Cancer,9.258024,NaN,False,True,False
6,ACH-000849,MDA-MB-468,breast,Breast Cancer,9.182568,2.649190,False,True,False
7,ACH-000109,NCI-H3255,lung,Lung Cancer,8.925347,2.465903,True,False,False
9,ACH-001649,Shmac 5,prostate,Prostate Cancer,8.362821,NaN,False,False,False
10,ACH-000865,KYSE-450,esophagus,Esophageal Cancer,8.360803,2.363419,True,False,False


## 5. Notes & known caveats

- **Column matching** (`get_rna_expression`, `get_protein_expression`) tries an
  exact gene-symbol match first (so `EGFR` won't accidentally match
  `EGFR-AS1`); it only falls back to a fuzzy/substring match if no exact
  match exists. A warning prints if more than one column matches.
- **File 6 (mutations)** identifies samples by `ProfileID`, not `ModelID` —
  unlike file 5. The loader resolves this internally via file 8, but a small
  number of `ProfileID`s have no entry in file 8 and are dropped (a warning
  reports how many).
- **Missing protein values are not "zero expression"** — `NaN` in
  `protein_expression` means the protein was not detected by mass spec in
  that cell line, which is different from confirmed absence.
- **Performance**: `_load_rna_matrix()` and `_load_protein_matrix()` load the
  *entire* wide matrix into memory on first call and cache it. The first
  query will be slower; subsequent queries for different genes reuse the
  cached matrix.
- This covers **Batch 1** only. Batch 2 (miRNA, metabolomics, global
  signatures, HPA, GEO) is not yet wired in — those need the Cellosaurus
  bridge (files 7/10/11) for HPA/GEO specifically.

---

# Batch 2 — miRNA, Metabolomics, Global Signatures

These three sources are wired in as **independent methods** (not yet merged
into `build_gene_profile()`), since they answer different kinds of
questions than the gene-expression query above:

| File | Method | ID resolution |
|---|---|---|
| 13 | `get_mirna_expression(mirna)` | CCLE_Name columns → ACH-ID via file 9 |
| 12 | `get_metabolite_level(metabolite)` | Ships with DepMap_ID directly |
| 14 | `get_global_signatures()` | ModelID present directly; de-duplicated to 1 row/cell line |

## 6. miRNA expression (File 13)

Column headers in file 13 use the old `CCLE_Name` format (e.g.
`DMS53_LUNG`), resolved here to `DepMap_ID` via file 9's `CCLE_Name` column.

**4 of 954 columns won't match exactly** — the loader prints the full list
on first call so you can inspect them (rather than silently dropping them).
Two are simple tissue-label mismatches between file 9 and file 13
(`KE97_STOMACH` vs `KE97_HAEMATOPOIETIC...`, `NCIH684_LARGE_INTESTINE` vs
`NCIH684_LIVER`); the other two (`COLO699_LUNG`, `NCIH1339_LUNG`) don't
appear in file 9 under any tissue label at all.

In [11]:
MIRNA = "hsa-miR-21"   # standard hsa-miR-xxx / hsa-let-7x naming

mirna = loader.get_mirna_expression(MIRNA)
print(f"miRNA expression rows: {len(mirna)}")
mirna.sort_values("mirna_expression", ascending=False).head()

[data_loader] Warning: 4 of 954 miRNA columns (CCLE_Name) had no exact match in file 9's CCLE_Name column and were dropped:
    - KE97_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE
    - NCIH1339_LUNG
    - NCIH684_LIVER
    - COLO699_LUNG
    These may be renamed/retired cell lines. Cross-check manually via file 7 (cellosaurus.csv) synonyms if you need them.
miRNA expression rows: 950


,DepMap_ID,mirna_expression
574,ACH-000621,188089.27
15,ACH-000648,79104.18
474,ACH-000182,70800.68
432,ACH-000720,56391.56
751,ACH-000213,55662.47


## 7. Metabolomics (File 12)

In [12]:
METABOLITE = "glutamine"

metab = loader.get_metabolite_level(METABOLITE)
print(f"Metabolomics rows: {len(metab)}")
metab.sort_values("metabolite_level", ascending=False).head()

Metabolomics rows: 927


,DepMap_ID,metabolite_level
367,ACH-000912,6.556283
194,ACH-000789,6.551007
256,ACH-000464,6.535858
23,ACH-000022,6.476222
518,ACH-000051,6.473314


In [13]:
# See all 225 available metabolite names
loader.list_metabolites()[:15]

['2-aminoadipate',
 '3-phosphoglycerate',
 'alpha-glycerophosphate',
 '4-pyridoxate',
 'aconitate',
 'adenine',
 'adipate',
 'alpha-ketoglutarate',
 'AMP',
 'citrate',
 'isocitrate',
 'CMP',
 'cystathionine',
 'cytidine',
 'dCMP']

## 8. Global genomic signatures (File 14)

One row per cell line: microsatellite instability (MSI), loss-of-heterozygosity
fraction, whole-genome doubling (WGD), chromosomal instability index (CIN),
ploidy and aneuploidy score. Useful as **exclusion criteria** — e.g. you may
want to flag or exclude highly unstable (high-CIN) cell lines depending on
your assay.

In [14]:
sigs = loader.get_global_signatures()
print(f"Cell lines with global signature data: {len(sigs)}")
sigs.head()

Cell lines with global signature data: 1955


,DepMap_ID,MSIScore,LoHFraction,WGD,CIN,Ploidy,Aneuploidy
0,ACH-000839,3.68,0.107443,1.0,0.502634,3.158291,20.0
1,ACH-000041,2.21,0.130089,1.0,0.523865,3.236089,19.0
2,ACH-002046,2.87,0.222342,1.0,0.679772,3.326715,30.0
3,ACH-002048,2.48,NaN,NaN,NaN,NaN,NaN
4,ACH-000042,2.07,NaN,NaN,NaN,NaN,NaN


In [15]:
# Example: flag cell lines with high chromosomal instability (CIN) for the
# current GENE's candidate list (from Section 4 above)
high_cin = sigs.loc[sigs["CIN"] > sigs["CIN"].quantile(0.9), "DepMap_ID"]

flagged = profile.merge(
    high_cin.to_frame("DepMap_ID").assign(high_CIN=True),
    on="DepMap_ID", how="left"
)
flagged["high_CIN"] = flagged["high_CIN"].fillna(False)

print(f"Of {len(flagged)} candidates for {GENE}, {flagged['high_CIN'].sum()} fall in the top 10% most chromosomally unstable cell lines.")
flagged.head(10)

Of 1434 candidates for EGFR, 129 fall in the top 10% most chromosomally unstable cell lines.


,DepMap_ID,cell_line_name,lineage,primary_disease,rna_expression,protein_expression,has_target_mutation,has_target_fusion,excluded,high_CIN
0,ACH-001523,HSC-1,skin,Skin Cancer,10.531869,NaN,False,False,False,False
1,ACH-000741,U-BLC1,urinary_tract,Bladder Cancer,10.076148,5.154394,False,True,False,False
2,ACH-000012,HCC827,lung,Lung Cancer,9.673645,2.724113,True,False,False,False
3,ACH-001411,UM-UC-5,urinary_tract,Bladder Cancer,9.417620,NaN,False,False,False,False
4,ACH-000637,KYSE-520,esophagus,Esophageal Cancer,9.286535,NaN,False,False,False,True
5,ACH-001864,YSCCC,bile_duct,Bile Duct Cancer,9.258024,NaN,False,True,False,False
6,ACH-000849,MDA-MB-468,breast,Breast Cancer,9.182568,2.649190,False,True,False,False
7,ACH-000109,NCI-H3255,lung,Lung Cancer,8.925347,2.465903,True,False,False,True
8,ACH-002680,170-MG-BA,central_nervous_system,Brain Cancer,8.867155,NaN,False,True,False,False
9,ACH-001649,Shmac 5,prostate,Prostate Cancer,8.362821,NaN,False,False,False,False


## 9. Notes — Batch 2

- **miRNA, metabolomics and global signatures are independent methods** —
  they are *not* merged into `build_gene_profile()`'s output. Combine them
  manually (as in the CIN example above) until a unified merge strategy is
  decided.
- **miRNA name matching is exact, case-insensitive** against file 13's
  `Description` column (standard `hsa-miR-xxx` naming). No fuzzy/partial
  matching — check spelling carefully.
- **Metabolite name matching is exact, case-insensitive** against file 12's
  column headers. Use `list_metabolites()` to see all 225 available names.
- **File 14 has duplicate ModelIDs** (cell lines re-sequenced multiple
  times); `get_global_signatures()` filters to the row flagged
  `IsDefaultEntryForModel == "Yes"` so each cell line appears once.